In [6]:
import pandas as pd

# Define the URL
url = "https://ourworldindata.org/grapher/cross-country-literacy-rates.csv?v=1&csvType=full&useColumnShortNames=false"

# Fetch the dataset with a custom User-Agent header
df = pd.read_csv(url, storage_options={'User-Agent': 'Our World In Data data fetch/1.0'})

# Display the first 5 rows and overview
print("First 5 Rows:")
print(df.head())

print("\nDataset Info:")
print(df.info())

First 5 Rows:
        Entity Code  Year  Literacy rate
0  Afghanistan  AFG  1950       3.000000
1  Afghanistan  AFG  1979      18.160000
2  Afghanistan  AFG  2011      31.450000
3  Afghanistan  AFG  2015      33.750000
4  Afghanistan  AFG  2020      36.014965

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2167 entries, 0 to 2166
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Entity         2167 non-null   object 
 1   Code           1890 non-null   object 
 2   Year           2167 non-null   int64  
 3   Literacy rate  2167 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 67.8+ KB
None


In [7]:
# 1. Clean column names (lowercase and replace spaces with underscores)
df.columns = df.columns.str.lower().str.replace(' ', '_')

# 2. Check entities that have NO country code (regional aggregates)
aggregates = df[df['code'].isna()]['entity'].unique()
print("Sample Regional Aggregates (No Code):", aggregates[:10])

# 3. Create a clean dataframe with ONLY individual countries (where code is present)
df_countries = df[df['code'].notna()].copy()

print(f"\nCleaned country-level dataset contains {len(df_countries)} rows.")
print(df_countries.head())

Sample Regional Aggregates (No Code): ['Central and Southern Asia (SDG)' 'Eastern and South-Eastern Asia (SDG)'
 'Europe and Northern America (SDG)'
 'Latin America and the Caribbean (SDG)'
 'Northern Africa and Western Asia (SDG)'
 'Oceania (excluding Australia and New Zealand) (SDG)'
 'Sub-Saharan Africa (SDG)' 'Western Europe (Buringh and van Zanden)']

Cleaned country-level dataset contains 1890 rows.
        entity code  year  literacy_rate
0  Afghanistan  AFG  1950       3.000000
1  Afghanistan  AFG  1979      18.160000
2  Afghanistan  AFG  2011      31.450000
3  Afghanistan  AFG  2015      33.750000
4  Afghanistan  AFG  2020      36.014965


In [8]:
# 1. Sort by country and year to guarantee chronological order
df_countries = df_countries.sort_values(by=['code', 'year'])

# 2. Extract earliest and latest records per country
country_summary = df_countries.groupby(['entity', 'code']).agg(
    first_year=('year', 'min'),
    first_rate=('literacy_rate', 'first'),
    latest_year=('year', 'max'),
    latest_rate=('literacy_rate', 'last')
).reset_index()

# 3. Calculate percentage point change
country_summary['point_change'] = (
    country_summary['latest_rate'] - country_summary['first_rate']
).round(2)

# 4. Filter for countries with more than 1 data point to see true progress
valid_progress = country_summary[country_summary['first_year'] != country_summary['latest_year']]

# Display top 10 most improved countries
top_improved = valid_progress.sort_values(by='point_change', ascending=False)
print("Top 10 Most Improved Countries in Literacy:")
print(top_improved[['entity', 'first_year', 'first_rate', 'latest_year', 'latest_rate', 'point_change']].head(10))

Top 10 Most Improved Countries in Literacy:
             entity  first_year  first_rate  latest_year  latest_rate  \
168          Poland        1475         0.0         1978        98.74   
100         Ireland        1475         0.0         1950        98.50   
201          Sweden        1475         1.0         1950        98.50   
196           Spain        1475         3.0         2021        99.70   
182    Saudi Arabia        1950         3.0         2024        97.93   
159            Oman        1950         3.0         2022        97.34   
219  United Kingdom        1475         5.0         1950        98.50   
171           Qatar        1950         7.5         2024        99.32   
72           France        1475         6.0         1950        96.50   
78          Germany        1475         9.0         1950        98.50   

     point_change  
168         98.74  
100         98.50  
201         97.50  
196         96.70  
182         94.93  
159         94.34  
219         

In [9]:
# Filter for modern era (1950 onward)
df_modern = df_countries[df_countries['year'] >= 1950].copy()

# Sort and aggregate modern progress
df_modern = df_modern.sort_values(by=['code', 'year'])
modern_summary = df_modern.groupby(['entity', 'code']).agg(
    start_year=('year', 'min'),
    start_rate=('literacy_rate', 'first'),
    latest_year=('year', 'max'),
    latest_rate=('literacy_rate', 'last')
).reset_index()

modern_summary['point_change'] = (modern_summary['latest_rate'] - modern_summary['start_rate']).round(2)

# Display top 10 modern educational transformations
top_modern = modern_summary.sort_values(by='point_change', ascending=False)
print("Top 10 Modern Literacy Transformations (1950–Present):")
print(top_modern[['entity', 'start_year', 'start_rate', 'latest_year', 'latest_rate', 'point_change']].head(10))

Top 10 Modern Literacy Transformations (1950–Present):
               entity  start_year  start_rate  latest_year  latest_rate  \
182      Saudi Arabia        1950         3.0         2024     97.93000   
159              Oman        1950         3.0         2022     97.34000   
171             Qatar        1950         7.5         2024     99.32000   
15            Bahrain        1950        12.5         2024     97.82000   
164  Papua New Guinea        1950         7.5         2022     87.00469   
110            Kuwait        1950        17.5         2020     96.46000   
227           Vietnam        1950        17.5         2022     96.13000   
97          Indonesia        1950        17.5         2020     96.00000   
106            Jordan        1950        17.5         2024     94.44000   
98               Iran        1950        12.5         2023     88.91682   

     point_change  
182         94.93  
159         94.34  
171         91.82  
15          85.32  
164         79.50  